In [ ]:
!pip install transformers==4.40.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.3
    Uninstalling transformers-4.52.3:
      Successfully uninstalled transformers-4.52.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:

!pip install datasets
!pip install accelerate
!pip install evaluate
!pip install sacrebleu

  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
data_files = {"train": "FinalProj_train.tsv", "valid": "FinalProj_valid.tsv", "test": "FinalProj_test.tsv"}
import pandas as pd
from datasets import Dataset, DatasetDict

# 파일명 정확히 반영
df_train = pd.read_csv("FInalProj_train.tsv", sep="\t")  # 대문자 I 사용
df_valid = pd.read_csv("FinalProj_valid.tsv", sep="\t")
df_test = pd.read_csv("FinalProj_test.tsv", sep="\t")

# Dataset으로 변환
ds_train = Dataset.from_pandas(df_train)
ds_valid = Dataset.from_pandas(df_valid)
ds_test = Dataset.from_pandas(df_test)

# DatasetDict로 묶기
dataset = DatasetDict({
    "train": ds_train,
    "validation": ds_valid,
    "test": ds_test
})

# 출력 확인
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['en', 'ko'],
        num_rows: 120000
    })
    validation: Dataset({
        features: ['en', 'ko'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['en', 'ko'],
        num_rows: 1000
    })
})
{'en': 'Four cities including Goyang City decided to collect the capacity to develop innovative local autonomous administration model based on strong local decentralization policy that selected exceptional city acquisition as a common task and differentiated local conditions and city characteristics.', 'ko': '고양시를 비롯한 4개 시는 특례시 쟁취를 공동의 과제로 선정하고 차별화된 지역 여건 및 도시 특성을 반영한 강력한 지방분권정책을 바탕으로 혁신적인 지역자치행정 모델 개발에 역량을 모으기로 했다.'}


In [ ]:
from transformers import AutoTokenizer
model_checkpoint = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")

max_length = 64

def preprocess_function(examples):
    inputs = examples["ko"]
    targets = examples["en"]
    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
import evaluate
metric = evaluate.load("sacrebleu")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:

import numpy as np

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # 모델이 logits 외 다른 걸 반환할 수도 있으므로 tuple 처리
    if isinstance(preds, tuple):
        preds = preds[0]

    # 숫자 → 문자열로 변환
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # -100 → pad_token_id로 변환 (디코딩을 위해)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # strip() 처리 + BLEU용 형식
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    # sacreBLEU 점수 계산
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"bleu": result["score"]}
from transformers import Seq2SeqTrainingArguments


In [ ]:
args = Seq2SeqTrainingArguments(
    "opusmt-ko2en",
    evaluation_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.40.1 --no-cache-dir


Found existing installation: transformers 4.40.1
Uninstalling transformers-4.40.1:
  Successfully uninstalled transformers-4.40.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 271.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.40.1 which is incompatible.


In [ ]:
!pip uninstall transformers evaluate -y
!pip uninstall peft sentence-transformers -y


Found existing installation: transformers 4.40.1
Uninstalling transformers-4.40.1:
  Successfully uninstalled transformers-4.40.1
Found existing installation: evaluate 0.4.3
Uninstalling evaluate-0.4.3:
  Successfully uninstalled evaluate-0.4.3
Found existing installation: peft 0.15.2
Uninstalling peft-0.15.2:
  Successfully uninstalled peft-0.15.2
Found existing installation: sentence-transformers 4.1.0
Uninstalling sentence-transformers-4.1.0:
  Successfully uninstalled sentence-transformers-4.1.0


In [ ]:
!rm -rf /usr/local/lib/python3.11/dist-packages/transformers
!rm -rf ~/.cache/huggingface/transformers


In [ ]:
pip install transformers==4.40.1 evaluate --no-cache-dir


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 269.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 286.6 MB/s eta 0:00:00


In [ ]:
from transformers import Seq2SeqTrainer  # 에러 안 나야 성공


In [ ]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()


Step,Training Loss
500,1.065800
1000,0.940700
1500,0.902400
2000,0.873200
2500,0.838200
3000,0.835800
3500,0.819500
4000,0.771400
4500,0.743900
5000,0.743900


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[65000]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[65000]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strate

TrainOutput(global_step=11250, training_loss=0.769517173936632, metrics={'train_runtime': 859.9271, 'train_samples_per_second': 418.64, 'train_steps_per_second': 13.083, 'total_flos': 6101705687040000.0, 'train_loss': 0.769517173936632, 'epoch': 3.0})

In [ ]:
from torch.utils.data import DataLoader

tokenized_datasets.set_format("torch")
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [ ]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
import re

def clean_text(text):
    # 문장부호 앞 공백 제거
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    return text.strip()

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [clean_text(pred) for pred in decoded_preds]
    decoded_labels = [[clean_text(label)] for label in decoded_labels]

    return decoded_preds, decoded_labels


In [ ]:
pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 115.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
metric.add_batch(predictions=decoded_preds, references=decoded_labels)


###training 및 bleu 스코어 출력

In [ ]:
from tqdm.auto import tqdm
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=128,
            )
        labels = batch["labels"]

        # Necessary to pad predictions and labels for being gathered
        generated_tokens = accelerator.pad_across_processes(
            generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
        )
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)

        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    results = metric.compute()
    print(f"epoch {epoch}, BLEU score: {results['score']:.2f}")

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)


  0%|          | 0/45000 [00:00<?, ?it/s]

  0%|          | 0/1125 [00:00<?, ?it/s]

epoch 0, BLEU score: 28.04


  0%|          | 0/1125 [00:00<?, ?it/s]

epoch 1, BLEU score: 29.71


  0%|          | 0/1125 [00:00<?, ?it/s]

epoch 2, BLEU score: 30.09


In [ ]:
model.eval()
input_text = "은행이 너무 멀어서 안되겠네요. 현찰이 필요하면 돈을 훔시세요"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=4,            # beam search로 더 나은 출력
    early_stopping=True,
    no_repeat_ngram_size=2
)


translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(translated)


The bank is too far. If you need cash, steal money.


###Inference 번역결과


In [ ]:
model.eval()

# 여러 문장을 리스트로 준비
input_texts = [
    "모든 액체, 젤, 에어로졸 등은 1커트짜리 여닫이 투명봉지 하나에 넣어야 합니다.",
    "미안하지만, 뒷쪽 아이들의 떠드는 소리가 커서, 광화문으로 가고 싶은데 표를 바꾸어 주시겠어요?",
    "은행이 너무 멀어서 안되겠네요. 현찰이 필요하면 돈을 훔시세요",
    "아무래도 분실한 것 같으니 분실 신고서를 작성해야 하겠습니다. 사무실로 같이 가실까요?",
    "부산에서 코로나 확진자가 급증해서 병상이 부족해지자 확진자 20명을 대구로 이송한다",
    "변기가 막혔습니다",
    "그 바지 좀 보여주십시오. 이거 얼마에 살 수 있는 것 입니까?",
    "비가 와서 백화점으로 가지 말고 두타로 갔으면 좋겠습니다.",
    "속이 안좋을 때는 죽이나 미음으로 아침을 대신합니다",
    "문대통령은 집단 이익에서 벗어나라고 말했다",
    "이것 좀 먹어 볼 몇 일 간의 시간을 주세요",
    "이 날 개미군단은 외인의 물량을 모두 받아 내었다",
    "통합 우승의 목표를 달성한 NC 다이노스 나성범이 메이저리그 진출이라는 또 다른 꿈을 향해 나아간다",
    "이번 구조 조정이 제품을 효과적으로 개발 하고 판매 하기 위한 회사의 능력 강화 조처임을 이해해 주시리라 생각합니다",
    "요즘 이 프로그램 녹화하며 많은 걸 느낀다",
]

# tokenizer와 모델에 입력
inputs = tokenizer(input_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

# 생성
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

# 디코딩
translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)

# 출력
for src, tgt in zip(input_texts, translations):
    print(f"[KOR] {src}\n[ENG] {tgt}\n")


[KOR] 모든 액체, 젤, 에어로졸 등은 1커트짜리 여닫이 투명봉지 하나에 넣어야 합니다.
[ENG] All liquids, gels and air-sols should be put into one transparent bag with one pack of margins.

[KOR] 미안하지만, 뒷쪽 아이들의 떠드는 소리가 커서, 광화문으로 가고 싶은데 표를 바꾸어 주시겠어요?
[ENG] I'm sorry, but the voices of the kids in the back are loud, so I want to go to Gwanghwamun, could you change the votes?

[KOR] 은행이 너무 멀어서 안되겠네요. 현찰이 필요하면 돈을 훔시세요
[ENG] The bank is too far. If you need cash, steal money.

[KOR] 아무래도 분실한 것 같으니 분실 신고서를 작성해야 하겠습니다. 사무실로 같이 가실까요?
[ENG] I think it's lost, so I need to fill out the loss report. Shall we go to the office together?

[KOR] 부산에서 코로나 확진자가 급증해서 병상이 부족해지자 확진자 20명을 대구로 이송한다
[ENG] As the number of Korona confirmed in Busan has soared, and the disease is not enough, 20 confirmed people will be transferred to Daegu.

[KOR] 변기가 막혔습니다
[ENG] The toilet is blocked.

[KOR] 그 바지 좀 보여주십시오. 이거 얼마에 살 수 있는 것 입니까?
[ENG] Show me those pants. How much can I buy?

[KOR] 비가 와서 백화점으로 가지 말고 두타로 갔으면 좋겠습니다.
[ENG] I hope you don't go to th